# P3 — Acquisition & Integration
**Research question:** Among films nominated for Best Picture for film years 2015–2024, do winners have higher IMDb audience ratings than non-winning nominees from the same film years?

**Sources:** [Wikipedia Best Picture](https://en.wikipedia.org/wiki/Academy_Award_for_Best_Picture) and official [IMDb non-commercial datasets](https://developer.imdb.com/non-commercial-datasets/).  

**Repository URL (fill in):** https://github.com/manika-17/dsa405-project


## 1. Acquisition plan and drift from A8
A8 proposed one cached Wikipedia article request and official IMDb `title.basics.tsv.gz` and `title.ratings.tsv.gz` downloads, with at least two seconds between requests. P2 already extracted and cleaned the Wikipedia nominee tables. P3 reproduces that collection with durable on-disk caching and reuses the P2 schema, title normalization, and winner-first ordering. IMDb remains an official download, **not** title-page scraping. This is an implementation refinement, not a change in sources. Any additional differences encountered during execution should be documented in the observations cell at the end.

The article has no pagination; IMDb's two published dataset files are separate resources rather than paginated API results. The code therefore explicitly records that pagination is not applicable. HTTP timeouts, non-200 responses, empty HTML tables, missing required columns, invalid files, and ambiguous matches cause informative errors rather than silent output.

Observed implementation drift: The IMDb lookup excludes ambiguous title/year combinations instead of forcing matches, and the left join preserves unmatched nominees. Five records remained unmatched in the last saved run; this is a documented matching limitation, not a change of data source. No IMDb page scraping was added.

In [1]:
from pathlib import Path
from io import StringIO
import re, time, unicodedata
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser
import pandas as pd
import requests

RAW = Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
PROCESSED = Path('data/processed'); PROCESSED.mkdir(parents=True, exist_ok=True)
WIKI_URL = 'https://en.wikipedia.org/wiki/Academy_Award_for_Best_Picture'
IMDB_URLS = {
    'basics': 'https://datasets.imdbws.com/title.basics.tsv.gz',
    'ratings': 'https://datasets.imdbws.com/title.ratings.tsv.gz',
}
# Before a first-time download, supply an actual contact address or Wikimedia user page.
# An honest descriptive identifier is used for already-cached re-runs.
HEADERS = {'User-Agent': 'NCSU-DSA405-BestPictureStudy/1.0 (educational coursework; contact: course instructor)'}
REQUEST_GAP_SECONDS = 2.0
_last_request = None

def throttle():
    global _last_request
    if _last_request is not None:
        time.sleep(max(0.0, REQUEST_GAP_SECONDS - (time.monotonic() - _last_request)))
    _last_request = time.monotonic()

def cached_download(url, path, *, timeout=(15, 120)):
    """One request per uncached resource; stream to temporary file and rename only on success."""
    path = Path(path)
    if path.is_file() and path.stat().st_size > 0:
        print(f'CACHE HIT: {path} ({path.stat().st_size:,} bytes)')
        return path
    if path.exists():
        raise ValueError(f'Invalid empty cache file: {path}; investigate before retrying')
    if urlparse(url).hostname == 'en.wikipedia.org' and 'contact: course instructor' in HEADERS['User-Agent']:
        raise ValueError('Before a new Wikimedia request, replace the User-Agent placeholder with an actual contact email or Wikimedia user page')
    throttle()
    temp = path.with_suffix(path.suffix + '.part')
    try:
        with requests.get(url, headers=HEADERS, stream=True, timeout=timeout) as response:
            response.raise_for_status()  # non-200 fails loudly
            with temp.open('wb') as f:
                for chunk in response.iter_content(chunk_size=1024*1024):
                    if chunk: f.write(chunk)
        if temp.stat().st_size == 0:
            raise ValueError(f'Empty response for {url}')
        temp.replace(path)
    except (requests.RequestException, OSError, ValueError) as exc:
        temp.unlink(missing_ok=True)
        raise RuntimeError(f'Acquisition failed for {url}: {exc}') from exc
    print(f'DOWNLOADED: {path} ({path.stat().st_size:,} bytes)')
    return path


### Permission and source guardrails
Wikipedia's article is attributed to Wikipedia contributors and its CC BY-SA 4.0 license; see [article](https://en.wikipedia.org/wiki/Academy_Award_for_Best_Picture), [license](https://creativecommons.org/licenses/by-sa/4.0/), and [terms](https://foundation.wikimedia.org/wiki/Policy:Terms_of_Use). IMDb permits limited personal/non-commercial use of its [official datasets](https://developer.imdb.com/non-commercial-datasets/) subject to [IMDb's conditions](https://help.imdb.com/article/imdb/general-information/can-i-use-imdb-data-in-my-software/G5JTRESSHJBBHTGX). **Information courtesy of IMDb (https://www.imdb.com). Used with permission.**

As in A8, this work uses public film-level data only: no login, paywall, personal viewer data, access-control bypass, proxies, or IMDb website scraping. A8 left current Wikipedia automated-access verification and IMDb dataset conditions as open checks. Confirm the current terms before first execution; if rules are unclear, ask the instructor rather than proceeding. The next cell checks Wikipedia robots rules and stops if the target path is disallowed or robots cannot be read.
Policy review (Nov 4, 2026): Wikimedia’s User-Agent policy requests an informative identifier with operator contact information; the existing generic “via course instructor” contact should be replaced with a real contact before a new request. IMDb’s official terms allow limited personal/noncommercial use of the published downloads, require attribution, and prohibit scraping IMDb website pages. The saved robots check returned True for the article, but this cached snapshot does not prove that future rules remain unchanged. Source links: https://foundation.wikimedia.org/wiki/Policy:Wikimedia_Foundation_User-Agent_Policy and https://help.imdb.com/article/imdb/general-information/can-i-use-imdb-data-in-my-software/G5JTRESSHJBBHTGX . Do not commit raw IMDb datasets or a republished IMDb-derived movie database to a public repository.

In [2]:
# Fetch robots.txt once to the same raw cache and parse it locally.
robots_path = cached_download('https://en.wikipedia.org/robots.txt', RAW/'wikipedia_robots.txt')
rp = RobotFileParser()
rp.parse(robots_path.read_text(encoding='utf-8').splitlines())
allowed = rp.can_fetch(HEADERS['User-Agent'], WIKI_URL)
print('Wikipedia article permitted by cached robots.txt:', allowed)
if not allowed:
    raise PermissionError('Wikipedia robots.txt disallows this User-Agent/article path. Stop and consult instructor.')
print('Robots cache is a snapshot: review live rules before first download if an older cache is reused.')

CACHE HIT: data/raw/wikipedia_robots.txt (28,275 bytes)
Wikipedia article permitted by cached robots.txt: True
Robots cache is a snapshot: review live rules before first download if an older cache is reused.


### Web acquisition: Wikipedia HTML (one page, no pagination)

In [3]:
wiki_path = cached_download(WIKI_URL, RAW/'wikipedia_best_picture.html')
html = wiki_path.read_text(encoding='utf-8')
if 'Best Picture' not in html:
    raise ValueError('Wikipedia response does not contain expected Best Picture text')
try:
    tables = pd.read_html(StringIO(html))
except ValueError as exc:
    raise ValueError('Wikipedia selector found no HTML tables; page structure may have changed') from exc
if not tables:
    raise ValueError('Wikipedia selector matched zero tables')
print('HTML tables found:', len(tables))

def flatten_columns(df):
    out = df.copy()
    out.columns = [' '.join(str(v) for v in c if str(v) != 'nan').strip() if isinstance(c, tuple)
                   else str(c).strip() for c in out.columns]
    return out

def find_exact_column(df, target):
    return next((c for c in df.columns if str(c).strip().lower() == target.lower()), None)

def select_award_tables(all_tables):
    """Fail loudly when required selectors are absent."""
    chosen = []
    for i, table in enumerate(all_tables):
        t = flatten_columns(table)
        film = find_exact_column(t, 'Film')
        year = find_exact_column(t, 'Year of film release')
        if film is not None and year is not None:
            chosen.append((i, t, film, year))
    if not chosen:
        raise ValueError('Selector matched nothing: no table with Film and Year of film release')
    return chosen

award_tables = select_award_tables(tables)
print('Best Picture table indexes:', [i for i, *_ in award_tables])
parts = []
for i, t, film, year in award_tables:
    x = pd.DataFrame({'film_year_raw': t[year].ffill(), 'title_raw': t[film], 'source_table': i})
    parts.append(x)
audit_raw = pd.concat(parts, ignore_index=True)
audit_raw['film_year'] = pd.to_numeric(audit_raw['film_year_raw'].astype(str).str.extract(r'(\d{4})')[0], errors='coerce')
project_raw = audit_raw.loc[audit_raw.film_year.between(2015, 2024)].copy()
if project_raw.empty:
    raise ValueError('No nominee records for 2015–2024; check Wikipedia structure')
print('Nominee rows before P2 cleaning:', len(project_raw))

CACHE HIT: data/raw/wikipedia_best_picture.html (1,013,451 bytes)
HTML tables found: 23
Best Picture table indexes: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Nominee rows before P2 cleaning: 91


### P2 cleaning, reproduced without modifying raw files
P2 trimmed whitespace, standardized film titles, removed duplicate title/year records if present, defined `award_year = film_year + 1`, and identified winners from the first-listed nominee for each film year. This notebook applies the same transformations, then validates the ten winner counts. If Wikipedia changes its ordering, the winner assumption must be reviewed rather than silently accepted.

In [4]:
nominees = project_raw.copy()
nominees['title'] = nominees['title_raw'].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
nominees = nominees.loc[nominees['title'].notna() & nominees['title'].ne('')].copy()
nominees['film_year'] = nominees['film_year'].astype(int)
duplicates_removed = int(nominees.duplicated(['film_year','title']).sum())
nominees = nominees.drop_duplicates(['film_year','title'], keep='first').sort_index().copy()

def normalize_title(series):
    return (series.astype('string').str.normalize('NFKC').str.lower()
            .str.replace(r'[’‘`]', "'", regex=True)
            .str.replace(r'[^a-z0-9]+', ' ', regex=True)
            .str.replace(r'\s+', ' ', regex=True).str.strip())
nominees['title_clean'] = normalize_title(nominees['title'])
nominees['award_year'] = nominees['film_year'] + 1
nominees['winner'] = ~nominees['film_year'].duplicated(keep='first')
nominees = nominees[['film_year','award_year','title','title_clean','winner']].reset_index(drop=True)
assert nominees.film_year.nunique() == 10, 'Missing one or more film years'
assert nominees.groupby('film_year').winner.sum().eq(1).all(), 'Expected exactly one winner per film year'
assert not nominees.duplicated(['film_year','title_clean']).any(), 'Normalized nominee keys are not unique'
print('P2-style cleaned nominees:', len(nominees), '| exact duplicates removed:', duplicates_removed)
display(nominees.groupby('film_year').agg(nominees=('title','size'), winners=('winner','sum')))

P2-style cleaned nominees: 91 | exact duplicates removed: 0


,nominees,winners
film_year,,
2015,8,1
2016,9,1
2017,9,1
2018,8,1
2019,9,1
2020,8,1
2021,10,1
2022,10,1
2023,10,1


### Official IMDb acquisition (no pagination; no API rate-limit headers)
IMDb's published files are **downloads**, not an API, so the assignment's API-only response-header requirement does not apply. The two distinct files are requested at most once, with a minimum two-second gap. The downloads are cached as the original `.gz` bytes under `data/raw/`. They are not modified. Do not use the incomplete gzip preview saved in A8: it is only a streamed sample, not a complete dataset.

In [5]:
basics_path = cached_download(IMDB_URLS['basics'], RAW/'title.basics.tsv.gz', timeout=(15, 300))
ratings_path = cached_download(IMDB_URLS['ratings'], RAW/'title.ratings.tsv.gz', timeout=(15, 300))
# Restrict to the columns needed for this research; source gzip files remain untouched.
basics = pd.read_csv(basics_path, sep='\t', compression='gzip', usecols=['tconst','titleType','primaryTitle','startYear'],
                     dtype='string', na_values=['\\N'], low_memory=False)
ratings = pd.read_csv(ratings_path, sep='\t', compression='gzip', usecols=['tconst','averageRating','numVotes'],
                      dtype={'tconst':'string','averageRating':'float64','numVotes':'Int64'}, na_values=['\\N'])
if basics.empty or ratings.empty:
    raise ValueError('IMDb file parsed to zero rows')
if basics.tconst.duplicated().any() or ratings.tconst.duplicated().any():
    raise ValueError('IMDb tconst identifiers unexpectedly duplicate')
print('IMDb basics:', len(basics), '| ratings:', len(ratings))

CACHE HIT: data/raw/title.basics.tsv.gz (229,783,200 bytes)
CACHE HIT: data/raw/title.ratings.tsv.gz (8,734,414 bytes)
IMDb basics: 12931235 | ratings: 1728747


## 2. Join design — stated before executing the merge

| Design item | Decision |
|---|---|
| Join keys | `film_year` and normalized `title_clean` (nominees), matched to `startYear` and normalized `primaryTitle` (IMDb); retain IMDb `tconst` as the unique film ID |
| Left grain | One row per Best Picture nominee per film year |
| Right grain | One row per uniquely matched IMDb movie title/year, with one `tconst` and its aggregate rating |
| Relationship | **One-to-one** for the accepted matches; ambiguous title/year candidates are excluded from the right lookup and reported |
| Expected output rows | **Exactly `len(nominees)`**, computed before merging; P1 estimated 91, but the actual P2-cleaned count is authoritative |
| Join type | **Left** join, so every Oscar nominee remains available for auditing |
| Unmatched | Remain in the output with missing IMDb fields; reported separately and excluded from rating-only comparisons until validated |
| Alternatives | An inner join could silently remove unmatched nominees (wreck). Joining on title alone could confuse remakes; allowing duplicate IMDb keys could multiply rows (fan-out). A left join with `validate='one_to_one'` protects against fan-out and preserves the left population. |

**Matching caution:** IMDb `startYear` can differ from the Oscar film year, and distinct films can share a normalized title/year. Do **not** automatically choose the first candidate or force matches. Inspect and document unresolved nominees in the generated audit CSV. Any manually verified `tconst` crosswalk must be justified and reviewed before adding it; this notebook does not invent one.

In [6]:
expected_rows = len(nominees)  # expectation set BEFORE merge
print('Expected merged row count:', expected_rows)
movies = basics.loc[basics.titleType.eq('movie')].copy()
movies['film_year'] = pd.to_numeric(movies.startYear, errors='coerce').astype('Int64')
movies = movies.loc[movies.film_year.between(2015, 2024)].copy()
movies['title_clean'] = normalize_title(movies.primaryTitle)
# Only keys unique among IMDb movie records can be used as an unambiguous lookup.
key_cols = ['film_year','title_clean']
ambiguous = movies.loc[movies.duplicated(key_cols, keep=False), key_cols + ['tconst','primaryTitle']].copy()
ambiguous.to_csv(PROCESSED/'ambiguous_imdb_candidates.csv', index=False)
unique_movies = movies.loc[~movies.duplicated(key_cols, keep=False), ['film_year','title_clean','tconst','primaryTitle']].copy()
unique_movies = unique_movies.merge(ratings, on='tconst', how='left', validate='one_to_one')
# P3 core join: preserve every Wikipedia nominee and explicitly enforce one-to-one.
merged = nominees.merge(unique_movies, on=key_cols, how='left', validate='one_to_one', indicator=True)
print('Join outcome:')
print(merged['_merge'].value_counts().to_string())
print('Nominees without a unique IMDb title/year match:', int(merged.tconst.isna().sum()))
print('Nominees matched to IMDb but missing ratings:', int((merged.tconst.notna() & merged.averageRating.isna()).sum()))
merged.loc[merged.tconst.isna(), ['film_year','title','title_clean']].to_csv(PROCESSED/'unmatched_nominees.csv', index=False)
display(merged.loc[merged.tconst.isna(), ['film_year','title']])

Expected merged row count: 91
Join outcome:
_merge
both          86
left_only      5
right_only     0
Nominees without a unique IMDb title/year match: 5
Nominees matched to IMDb but missing ratings: 0


,film_year,title
0,2015,Spotlight
44,2020,The Father
45,2020,Judas and the Black Messiah
49,2020,Sound of Metal
77,2023,Maestro


## 3. Verification suite
Each assertion below documents the assumption it protects. These checks test **structural correctness**, not whether every IMDb match is semantically correct. Unmatched rows and title/year differences require human review. No match rate is fabricated.

In [7]:

# Protects against fan-out: the merge must preserve all nominees.
assert len(merged) == expected_rows, "Join changed the row count"

# Protects against duplicate nominees after joining.
assert not merged.duplicated(
    ["film_year", "title_clean"]
).any(), "Duplicate nominee keys found"

# Protects against invalid IMDb ratings.
assert merged["averageRating"].dropna().between(
    1, 10
).all(), "IMDb ratings outside valid range"

# Protects against incorrect winner counts.
assert merged.groupby("film_year")["winner"].sum().eq(
    1
).all(), "Expected one winner per film year"

# Protects against lost nominee records.
assert merged["_merge"].ne("right_only").all(), \
    "Unexpected right-only records"

print("All five post-merge verification checks passed.")


All five post-merge verification checks passed.


**Winner-label verification against the published Best Picture winners**
The one-winner-per-year assertion alone cannot tell whether the correct nominee was labeled. This independent reference list checks the ten winners for film years 2015–2024. If Wikipedia changes its order, this check fails rather than silently assigning a different winner.

In [8]:
# Reference winners for the FILM years, not ceremony years.
expected_winners = {
    2015: 'Spotlight', 2016: 'Moonlight', 2017: 'The Shape of Water',
    2018: 'Green Book', 2019: 'Parasite', 2020: 'Nomadland',
    2021: 'CODA', 2022: 'Everything Everywhere All at Once',
    2023: 'Oppenheimer', 2024: 'Anora',
}
actual_winners = nominees.loc[nominees['winner']].set_index('film_year')['title'].to_dict()
for year, title in expected_winners.items():
    assert normalize_title(pd.Series([actual_winners.get(year, '')])).iloc[0] == normalize_title(pd.Series([title])).iloc[0], (year, actual_winners.get(year), title)
print('Winner identity checks passed for all 10 film years.')
display(pd.DataFrame({'film_year': list(expected_winners), 'verified_winner': list(expected_winners.values())}))

Winner identity checks passed for all 10 film years.


,film_year,verified_winner
0,2015,Spotlight
1,2016,Moonlight
2,2017,The Shape of Water
3,2018,Green Book
4,2019,Parasite
5,2020,Nomadland
6,2021,CODA
7,2022,Everything Everywhere All at Once
8,2023,Oppenheimer
9,2024,Anora


**Diagnose the five unmatched nominees (do not force a match)**
For each unmatched nominee, inspect same-title IMDb movie candidates and their release years. A candidate is not automatically a verified identity match. This produces a review file so the cause can be documented accurately after running with cached IMDb data.

In [9]:
unmatched = merged.loc[merged['tconst'].isna(), ['film_year','title','title_clean']].copy()
# Inspect ALL movie years, not just the 2015–2024 join window.
all_movies = basics.loc[basics.titleType.eq('movie'), ['tconst','primaryTitle','startYear']].copy()
all_movies['title_clean'] = normalize_title(all_movies['primaryTitle'])
candidates = unmatched.merge(all_movies, on='title_clean', how='left')
candidates['candidate_start_year'] = pd.to_numeric(candidates['startYear'], errors='coerce')
candidates['year_difference'] = candidates['candidate_start_year'] - candidates['film_year']
candidates['same_year'] = candidates['year_difference'].eq(0)
candidates = candidates.sort_values(['film_year','title','same_year','year_difference'], ascending=[True,True,False,True], na_position='last')
candidates.to_csv(PROCESSED/'unmatched_candidate_review.csv', index=False)
for (year,title), group in candidates.groupby(['film_year','title'], sort=True):
    print(f'\n{year}: {title} — {len(group)} same-title candidate row(s)')
    display(group[['tconst','primaryTitle','candidate_start_year','year_difference','same_year']].head(12))
print('Saved unmatched_candidate_review.csv; inspect candidate identities before accepting any manual crosswalk.')


2015: Spotlight — 2 same-title candidate row(s)


,tconst,primaryTitle,candidate_start_year,year_difference,same_year
0,tt1895587,Spotlight,2015,0,True
1,tt7785302,Spotlight,2015,0,True



2020: Judas and the Black Messiah — 1 same-title candidate row(s)


,tconst,primaryTitle,candidate_start_year,year_difference,same_year
15,tt9784798,Judas and the Black Messiah,2021,1,False



2020: Sound of Metal — 1 same-title candidate row(s)


,tconst,primaryTitle,candidate_start_year,year_difference,same_year
16,tt5363618,Sound of Metal,2019,-1,False



2020: The Father — 13 same-title candidate row(s)


,tconst,primaryTitle,candidate_start_year,year_difference,same_year
6,tt10272386,The Father,2020,0,True
10,tt16531196,The Father,2020,0,True
5,tt0296967,The Father,1947,-73,False
2,tt0069752,The Father,1971,-49,False
4,tt0148324,The Father,1975,-45,False
12,tt5049400,The Father,1979,-41,False
3,tt0117315,The Father,1996,-24,False
9,tt13765040,The Father,2015,-5,False
7,tt10409554,The Father,2019,-1,False
11,tt34392970,The Father,2026,6,False



2023: Maestro — 13 same-title candidate row(s)


,tconst,primaryTitle,candidate_start_year,year_difference,same_year
21,tt21970768,Maestro,2023,0,True
28,tt5535276,Maestro,2023,0,True
17,tt0368851,Maestro,2002,-21,False
18,tt11355356,Maestro,2002,-21,False
20,tt1785457,Maestro,2009,-14,False
23,tt3476480,Maestro,2014,-9,False
22,tt2261817,Maestro,2015,-8,False
24,tt3505800,Maestro!,2015,-8,False
19,tt13130400,Maestro,2021,-2,False
26,tt37179751,Maestro,2025,2,False


Saved unmatched_candidate_review.csv; inspect candidate identities before accepting any manual crosswalk.


### Required demonstration: a failure caught loudly
The test below deliberately supplies a table with the wrong column names. The selector must raise `ValueError`; the test itself asserts that it did. This does not trigger another network request or alter raw data.

In [10]:
try:
    select_award_tables([pd.DataFrame({'Not Film': ['x'], 'Not Year': [2020]})])
except ValueError as exc:
    assert 'Selector matched nothing' in str(exc)
    print('EXPECTED FAILURE CAUGHT LOUDLY:', type(exc).__name__, '-', exc)
else:
    raise AssertionError('FAIL: selector returned silently on malformed input')

EXPECTED FAILURE CAUGHT LOUDLY: ValueError - Selector matched nothing: no table with Film and Year of film release


### Additional fan-out demonstration
A duplicate right-side key should cause pandas to reject a one-to-one merge. This uses tiny synthetic data, not a modified source file.

In [11]:
try:
    pd.DataFrame({'key':[1]}).merge(pd.DataFrame({'key':[1,1]}), on='key', validate='one_to_one')
except pd.errors.MergeError as exc:
    print('EXPECTED JOIN FAILURE CAUGHT:', type(exc).__name__, '-', exc)
else:
    raise AssertionError('FAIL: duplicate right keys were not rejected')

EXPECTED JOIN FAILURE CAUGHT: MergeError - Merge keys are not unique in right dataset; not a one-to-one merge


## 4. Saved outputs and finding
The merged table and audit files are **derived** data in `data/processed/`. The HTML and IMDb `.gz` files in `data/raw/` remain unchanged. The unmatched list is an important finding, not an invitation to make up ratings.

In [12]:
merged.to_csv(PROCESSED/'best_picture_imdb_joined.csv', index=False)
summary = pd.DataFrame([{
    'nominee_rows':len(nominees), 'joined_rows':len(merged),
    'matched_imdb_titles':int(merged.tconst.notna().sum()),
    'unmatched_titles':int(merged.tconst.isna().sum()),
    'rated_nominees':int(merged.averageRating.notna().sum()),
    'ambiguous_imdb_candidate_rows':len(ambiguous),
}])
summary.to_csv(PROCESSED/'join_summary.csv', index=False)
display(summary)
print('Saved processed files to', PROCESSED.resolve())

,nominee_rows,joined_rows,matched_imdb_titles,unmatched_titles,rated_nominees,ambiguous_imdb_candidate_rows
0,91,91,86,5,86,6357


Saved processed files to /content/data/processed


### One thing learned or checked
One thing that surprised me was that even after normalizing film titles and matching on release year, five of the 91 Best Picture nominees did not match uniquely to IMDb. These included *Spotlight*, *The Father*, *Judas and the Black Messiah*, *Sound of Metal*, and *Maestro*. The join successfully preserved all 91 nominees, but only 86 received IMDb ratings, resulting in a match rate of approximately 94.5%. This showed me that even when two sources contain information about the same films, differences in release years or duplicate title records can make integration more complicated than expected. Rather than forcing potentially incorrect matches, I kept these records unmatched for further review.

### AI use disclosure
I used ChatGPT to help implement acquisition safeguards and join verification checks.

## 5. Self-scored P3 rubric
| Criterion | Weight | Self-score (0–3) | Evidence and remaining limitation |
|---|---:|---:|---|
| Acquisition code | ×2 | 3/3 | Cached original responses; two-second request gap; timeout, HTTP and selector handling; demonstrated selector failure in saved run. |
| Ethics & legality | ×2 | 3/3 | A8 consistency, drift, source links and noncommercial attribution; operator contact must be supplied before any NEW Wikipedia request. |
| Join design | ×2 | 3/3 | Keys, grains, one-to-one cardinality, expected 91, left join, `validate`, alternatives. |
| Verification suite | ×3 | 3/3 | the five assertions, winner checks, and unmatched-film diagnostics executed successfully. |
| Reproducibility & readability | ×1 | 2/3 | README and deterministic code; full restarted-kernel execution of this REVISED notebook is still pending. |
| **Self-assessed weighted total** | **×10** | **29/30** | **Not an instructor grade; contingent on a clean run and repository publication.** |
